# MobileNetV2 Transfer Learning — Cochleagram / Log-Mel-Spectrogram / Spectrogram

This notebook trains **MobileNetV2** separately on each of the three spectrogram
representations in the dataset (`cochleagram`, `log-mel-spectrograms`, `spectrograms`),
**one representation at a time**, using the exact experimental setup:

| Hyperparameter | Value |
|---|---|
| Batch size | 128 |
| Learning rate | 1e-3 |
| Optimizer | Adam |
| Gradient decay factor (Adam β₁) | 0.9 |
| Epochs | 30 |
| Train / Val / Test split | 70% / 15% / 15% |

**Assumptions made about the dataset layout** (based on the Data Explorer screenshot),
please double check `CONFIG["data_root"]` and the manifest column names in Cell 3 —
everything else adapts automatically once those two things are correct:

```
Preprocessed Dataset/
├── cochleagram/
│   ├── cochleagram/                 <- .npy files
│   └── cochleagram_manifest.csv     <- filename -> label mapping
├── log-mel-spectrograms/
│   ├── log-mel-spectrograms/
│   └── log-mel-spectrograms_manifest.csv
└── spectrograms/
    ├── spectrograms/
    └── spectrograms_manifest.csv
```

The code is split into small, single-purpose functions so every step (data loading,
model building, training, evaluation) can be read and reused independently.

In [ ]:
# ------------------------- Imports -------------------------
import os
import time
import json
import copy
import numpy as np
import pandas as pd
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

In [ ]:
# ------------------------- Config -------------------------
# EDIT THIS CELL to match your actual Kaggle dataset path & manifest columns.

CONFIG = {
    # Root folder that contains cochleagram/, log-mel-spectrograms/, spectrograms/
    "data_root": "/kaggle/input/<your-dataset-slug>/Preprocessed Dataset",

    # The three spectrogram types to train on, one at a time.
    # Folder-name convention assumed: data_root/<type>/<type>/*.npy
    #                                  data_root/<type>/<type>_manifest.csv
    "spectrogram_types": ["cochleagram", "log-mel-spectrograms", "spectrograms"],

    # Manifest CSV column names — change these if your columns are named differently.
    "filename_col": "filename",
    "label_col": "label",

    # Experimental setup (fixed, as specified)
    "batch_size": 128,
    "learning_rate": 1e-3,
    "adam_beta1": 0.9,        # "gradient decay factor of 0.9" == Adam's beta_1 (momentum term)
    "adam_beta2": 0.999,      # left at PyTorch default, not specified in the paper
    "epochs": 30,
    "train_ratio": 0.70,
    "val_ratio": 0.15,
    "test_ratio": 0.15,

    # Model input size expected by MobileNetV2
    "img_size": 224,

    "num_workers": 2,
    "output_dir": "/kaggle/working",
}

os.makedirs(CONFIG["output_dir"], exist_ok=True)
CONFIG

In [ ]:
# ------------------------- Manifest & path helpers -------------------------

def get_type_paths(spectrogram_type, config=CONFIG):
    """Return (npy_dir, manifest_path) for a given spectrogram type."""
    type_root = os.path.join(config["data_root"], spectrogram_type)
    npy_dir = os.path.join(type_root, spectrogram_type)
    manifest_path = os.path.join(type_root, f"{spectrogram_type}_manifest.csv")
    return npy_dir, manifest_path


def load_manifest(spectrogram_type, config=CONFIG):
    """Load manifest.csv for a spectrogram type and sanity-check columns."""
    _, manifest_path = get_type_paths(spectrogram_type, config)
    df = pd.read_csv(manifest_path)

    fcol, lcol = config["filename_col"], config["label_col"]
    if fcol not in df.columns or lcol not in df.columns:
        raise ValueError(
            f"Expected columns '{fcol}' and '{lcol}' in {manifest_path}, "
            f"but found: {list(df.columns)}. Update CONFIG['filename_col'] / "
            f"CONFIG['label_col'] accordingly."
        )
    return df[[fcol, lcol]].rename(columns={fcol: "filename", lcol: "label"})

In [ ]:
# ------------------------- Splitting -------------------------

def make_splits(df, config=CONFIG, seed=SEED):
    """Stratified 70/15/15 train/val/test split."""
    train_df, temp_df = train_test_split(
        df,
        test_size=(config["val_ratio"] + config["test_ratio"]),
        stratify=df["label"],
        random_state=seed,
    )
    val_frac_of_temp = config["val_ratio"] / (config["val_ratio"] + config["test_ratio"])
    val_df, test_df = train_test_split(
        temp_df,
        test_size=(1 - val_frac_of_temp),
        stratify=temp_df["label"],
        random_state=seed,
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

In [ ]:
# ------------------------- Dataset -------------------------

class SpectrogramDataset(Dataset):
    """Loads a single .npy spectrogram, normalizes it, resizes it to the
    size MobileNetV2 expects, and replicates it to 3 channels."""

    IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def __init__(self, df, npy_dir, label_encoder, img_size=224):
        self.df = df.reset_index(drop=True)
        self.npy_dir = npy_dir
        self.label_encoder = label_encoder
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def _load_array(self, filename):
        path = os.path.join(self.npy_dir, filename)
        if not path.endswith(".npy"):
            path = path + ".npy"
        arr = np.load(path).astype(np.float32)
        if arr.ndim == 3:  # already has a channel dim, e.g. (H, W, C)
            arr = arr[..., 0]
        return arr

    def _to_model_input(self, arr):
        # per-sample min-max normalization to [0, 1]
        amin, amax = arr.min(), arr.max()
        arr = (arr - amin) / (amax - amin + 1e-8)

        arr = cv2.resize(arr, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        arr = np.repeat(arr[..., None], 3, axis=-1)  # (H, W, 3)

        arr = (arr - self.IMAGENET_MEAN) / self.IMAGENET_STD
        arr = np.transpose(arr, (2, 0, 1))  # -> (3, H, W)
        return torch.from_numpy(arr.astype(np.float32))

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        arr = self._load_array(row["filename"])
        x = self._to_model_input(arr)
        y = int(self.label_encoder.transform([row["label"]])[0])
        return x, y

In [ ]:
# ------------------------- Dataloaders -------------------------

def build_dataloaders(spectrogram_type, config=CONFIG, seed=SEED):
    npy_dir, _ = get_type_paths(spectrogram_type, config)
    df = load_manifest(spectrogram_type, config)

    label_encoder = LabelEncoder().fit(df["label"])
    num_classes = len(label_encoder.classes_)

    train_df, val_df, test_df = make_splits(df, config, seed)

    train_ds = SpectrogramDataset(train_df, npy_dir, label_encoder, config["img_size"])
    val_ds = SpectrogramDataset(val_df, npy_dir, label_encoder, config["img_size"])
    test_ds = SpectrogramDataset(test_df, npy_dir, label_encoder, config["img_size"])

    loader_kwargs = dict(batch_size=config["batch_size"], num_workers=config["num_workers"], pin_memory=True)
    train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

    print(f"[{spectrogram_type}] classes={num_classes} | train={len(train_df)} "
          f"val={len(val_df)} test={len(test_df)}")

    return train_loader, val_loader, test_loader, num_classes, label_encoder

In [ ]:
# ------------------------- Model -------------------------

def build_model(num_classes, config=CONFIG):
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model.to(DEVICE)

In [ ]:
# ------------------------- Train / Eval loops -------------------------

def run_epoch(model, loader, criterion, optimizer=None):
    """One pass over `loader`. If optimizer is given, trains; else evaluates."""
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, all_preds, all_labels = 0.0, [], []
    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            if is_train:
                optimizer.zero_grad()

            logits = model(x)
            loss = criterion(logits, y)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * x.size(0)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(y.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    return avg_loss, acc, all_preds, all_labels


def train_model(model, train_loader, val_loader, config=CONFIG):
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=config["learning_rate"],
        betas=(config["adam_beta1"], config["adam_beta2"]),
    )

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_loss = float("inf")
    best_state = None

    for epoch in range(1, config["epochs"] + 1):
        t0 = time.time()
        train_loss, train_acc, _, _ = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _, _ = run_epoch(model, val_loader, criterion, optimizer=None)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())

        print(f"Epoch {epoch:02d}/{config['epochs']} | "
              f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
              f"{time.time() - t0:.1f}s")

    model.load_state_dict(best_state)
    return model, history

In [ ]:
# ------------------------- Reporting -------------------------

def plot_history(history, title, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    axes[0].plot(history["train_loss"], label="train")
    axes[0].plot(history["val_loss"], label="val")
    axes[0].set_title(f"{title} — Loss")
    axes[0].set_xlabel("epoch"); axes[0].legend()

    axes[1].plot(history["train_acc"], label="train")
    axes[1].plot(history["val_acc"], label="val")
    axes[1].set_title(f"{title} — Accuracy")
    axes[1].set_xlabel("epoch"); axes[1].legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120)
    plt.show()


def evaluate_on_test(model, test_loader, label_encoder, title):
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc, preds, labels = run_epoch(model, test_loader, criterion, optimizer=None)
    f1 = f1_score(labels, preds, average="macro")

    print(f"\n=== {title} — TEST RESULTS ===")
    print(f"loss={test_loss:.4f}  accuracy={test_acc:.4f}  macro_f1={f1:.4f}\n")
    print(classification_report(labels, preds, target_names=label_encoder.classes_))

    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(5, 4))
    plt.imshow(cm, cmap="Blues")
    plt.title(f"{title} — Confusion Matrix")
    plt.colorbar()
    plt.xlabel("Predicted"); plt.ylabel("True")
    plt.tight_layout()
    plt.show()

    return {"test_loss": test_loss, "test_acc": test_acc, "test_f1_macro": f1}

In [ ]:
# ------------------------- Single-experiment runner -------------------------

def run_experiment(spectrogram_type, config=CONFIG, seed=SEED):
    """Train + evaluate MobileNetV2 on ONE spectrogram type end-to-end."""
    print(f"\n{'=' * 70}\nTraining on: {spectrogram_type}\n{'=' * 70}")

    train_loader, val_loader, test_loader, num_classes, label_encoder = build_dataloaders(
        spectrogram_type, config, seed
    )

    model = build_model(num_classes, config)
    model, history = train_model(model, train_loader, val_loader, config)

    plot_history(history, title=spectrogram_type,
                 save_path=os.path.join(config["output_dir"], f"{spectrogram_type}_history.png"))

    test_metrics = evaluate_on_test(model, test_loader, label_encoder, title=spectrogram_type)

    ckpt_path = os.path.join(config["output_dir"], f"mobilenetv2_{spectrogram_type}.pt")
    torch.save(model.state_dict(), ckpt_path)
    print(f"Saved checkpoint -> {ckpt_path}")

    return {"history": history, "test_metrics": test_metrics, "checkpoint": ckpt_path}

In [ ]:
# ------------------------- Run all three, one at a time -------------------------

results = {}
for spectrogram_type in CONFIG["spectrogram_types"]:
    results[spectrogram_type] = run_experiment(spectrogram_type, CONFIG)

In [ ]:
# ------------------------- Final comparison -------------------------

summary_df = pd.DataFrame({
    spectrogram_type: res["test_metrics"]
    for spectrogram_type, res in results.items()
}).T
summary_df.index.name = "spectrogram_type"
print(summary_df)

summary_df.to_csv(os.path.join(CONFIG["output_dir"], "results_summary.csv"))
with open(os.path.join(CONFIG["output_dir"], "results_summary.json"), "w") as f:
    json.dump({k: v["test_metrics"] for k, v in results.items()}, f, indent=2)

summary_df